In [11]:
from langchain_core.documents import Document


In [24]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from pathlib import Path

In [25]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: Baru.pdf


invalid pdf header: b'PK\x03\x04\x14'
EOF marker not found


  ✓ Loaded 1 pages

Processing: batch15.pdf
  ✓ Loaded 1 pages

Processing: cluco.pdf
  ✗ Error: Stream has ended unexpectedly

Total documents loaded: 2


In [26]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-01-28T16:12:41+05:30', 'author': 'venu babu', 'moddate': '2026-01-28T16:12:41+05:30', 'source': '..\\data\\pdf\\Baru.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Baru.pdf', 'file_type': 'pdf'}, page_content='MEDICAL CERTIFICATE \n \nThis is to certify that Ms. Bhargavi has been suffering from \nsevere abdominal pain and fever. Upon medical evaluation and \ndiagnostic investigations, she was found to have a stone in the \nright kidney associated with infection. \nShe is currently undergoing treatment to control and cure \nthe infection. After successful management of the infection, \nshe is advised to undergo surgical removal of the kidney stone \nafter approximately 20 days. \nUntil the scheduled surgery, the patient is not advised to \ntravel, as it may aggravate her condition. Post-surgery, she will \nrequire at least one week of complete rest. Aft

In [27]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [32]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [33]:
chunks=split_documents(all_pdf_documents)
chunks

Split 2 documents into 3 chunks

Example chunk:
Content: MEDICAL CERTIFICATE 
 
This is to certify that Ms. Bhargavi has been suffering from 
severe abdominal pain and fever. Upon medical evaluation and 
diagnostic investigations, she was found to have a st...
Metadata: {'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-01-28T16:12:41+05:30', 'author': 'venu babu', 'moddate': '2026-01-28T16:12:41+05:30', 'source': '..\\data\\pdf\\Baru.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Baru.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-01-28T16:12:41+05:30', 'author': 'venu babu', 'moddate': '2026-01-28T16:12:41+05:30', 'source': '..\\data\\pdf\\Baru.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Baru.pdf', 'file_type': 'pdf'}, page_content='MEDICAL CERTIFICATE \n \nThis is to certify that Ms. Bhargavi has been suffering from \nsevere abdominal pain and fever. Upon medical evaluation and \ndiagnostic investigations, she was found to have a stone in the \nright kidney associated with infection. \nShe is currently undergoing treatment to control and cure \nthe infection. After successful management of the infection, \nshe is advised to undergo surgical removal of the kidney stone \nafter approximately 20 days. \nUntil the scheduled surgery, the patient is not advised to \ntravel, as it may aggravate her condition. Post-surgery, she will \nrequire at least one week of complete rest. Aft

In [12]:
doc=Document(
    page_content="This is a test document",
              metadata={"source": "test.txt",
                        "page":1,
                        'author':'venu' })
doc

Document(metadata={'source': 'test.txt', 'page': 1, 'author': 'venu'}, page_content='This is a test document')

In [13]:
import os
os.makedirs('../data/text_files', exist_ok=True)

In [14]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [15]:
from langchain_community.document_loaders import TextLoader
loader=TextLoader('../data/text_files/python_intro.txt',encoding='utf-8')
documents=loader.load()
print(documents)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [16]:
from langchain_community.document_loaders import DirectoryLoader
di=DirectoryLoader("../data/text_files",
                   glob="*.txt",
                   loader_cls=TextLoader,
                   loader_kwargs={'encoding':'utf-8'},
                   show_progress=False)
do=di.load()
do


[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popu

In [17]:
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
pd=DirectoryLoader("../data/pdf",
                   glob="*.pdf",
                   loader_cls=PyMuPDFLoader,
                   show_progress=False)
pdf=pd.load()
pdf


[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-01-28T16:12:41+05:30', 'source': '..\\data\\pdf\\Baru.pdf', 'file_path': '..\\data\\pdf\\Baru.pdf', 'total_pages': 1, 'format': 'PDF 1.7', 'title': '', 'author': 'venu babu', 'subject': '', 'keywords': '', 'moddate': '2026-01-28T16:12:41+05:30', 'trapped': '', 'modDate': "D:20260128161241+05'30'", 'creationDate': "D:20260128161241+05'30'", 'page': 0}, page_content='MEDICAL CERTIFICATE \n \nThis is to certify that Ms. Bhargavi has been suffering from \nsevere abdominal pain and fever. Upon medical evaluation and \ndiagnostic investigations, she was found to have a stone in the \nright kidney associated with infection. \nShe is currently undergoing treatment to control and cure \nthe infection. After successful management of the infection, \nshe is advised to undergo surgical removal of the kidney stone \nafter approximately 20 days. \nUntil the scheduled surgery, the patient 

In [18]:
type(pdf[0])

langchain_core.documents.base.Document

In [3]:
import numpy as np
from  sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List ,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\venup\OneDrive\Desktop\Rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1807.34it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


In [19]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [34]:
chunks

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-01-28T16:12:41+05:30', 'author': 'venu babu', 'moddate': '2026-01-28T16:12:41+05:30', 'source': '..\\data\\pdf\\Baru.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Baru.pdf', 'file_type': 'pdf'}, page_content='MEDICAL CERTIFICATE \n \nThis is to certify that Ms. Bhargavi has been suffering from \nsevere abdominal pain and fever. Upon medical evaluation and \ndiagnostic investigations, she was found to have a stone in the \nright kidney associated with infection. \nShe is currently undergoing treatment to control and cure \nthe infection. After successful management of the infection, \nshe is advised to undergo surgical removal of the kidney stone \nafter approximately 20 days. \nUntil the scheduled surgery, the patient is not advised to \ntravel, as it may aggravate her condition. Post-surgery, she will \nrequire at least one week of complete rest. Aft

In [35]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 3 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.61it/s]

Generated embeddings with shape: (3, 384)
Adding 3 documents to vector store...
Successfully added 3 documents to vector store
Total documents in collection: 3


In [36]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [37]:
rag_retriever

In [39]:
rag_retriever.retrieve("What is abstraction of Real-Time Stock Prediction Using an Improved LSTM DNN Architecture for Forecasting")

Retrieving documents for query: 'What is abstraction of Real-Time Stock Prediction Using an Improved LSTM DNN Architecture for Forecasting'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.91it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_a60b1bef_2',
  'content': "Abstract : \nStock markets behave as complex nonlinear systems. Traditional ML \nmodels fail to capture dependencies, while LSTM networks improve \nsequence modeling. However, most studies predict only the next day's \nprice and lack robustness. This project proposes a new hybrid model \ncombining LSTM, Attention, and DNN layers. \n \n \nBatch Number – 15      Project Guide: \nP . V. Gowtham Reddy Y22CD129                                    Mr.A V Krishnarao Padyala \nP . Venu Babu Y22CD135 \nT. Revathipathi Y22CD167",
  'metadata': {'creator': 'Microsoft® Word 2021',
   'page': 0,
   'source_file': 'batch15.pdf',
   'source': '..\\data\\pdf\\batch15.pdf',
   'author': 'veerendra kumar',
   'file_type': 'pdf',
   'moddate': '2025-12-08T12:03:06+05:30',
   'total_pages': 1,
   'content_length': 511,
   'page_label': '1',
   'producer': 'Microsoft® Word 2021',
   'creationdate': '2025-12-08T12:03:06+05:30',
   'doc_index': 2},
  'similarity_score':